# 04 · Break the detector (≈ 15 min, small groups)
**Challenge 5.** Find answers where the detector is wrong or overconfident, and say *why*.

Everything here runs offline on the stored results across all five settings.

In [ ]:
import sys, pathlib, warnings
from collections import Counter
REPO = "pycones2026-uncertainty-workshop"   # <- must match your GitHub repo name (used on Colab only)
if "google.colab" in sys.modules:
    ROOT = pathlib.Path("/content") / REPO
    if not ROOT.exists():
        import subprocess
        subprocess.run(["git", "clone", f"https://github.com/Shafiya0101/{REPO}.git", str(ROOT)], check=True)
else:
    ROOT = pathlib.Path.cwd()
    if ROOT.name in ("notebooks", "solutions"):
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from uncertainty_lib import *
from uncertainty_lib import checks
from uncertainty_lib.plotting import set_style, plot_region_grid, plot_risk_coverage
set_style()
pd.set_option("display.max_colwidth", 70)

In [ ]:
frames = [add_signals(load_setting(s)) for s in SETTINGS]        # normalize per setting
allq = pd.concat(frames, ignore_index=True)
allq = pd.concat([add_regions(f) for f in frames], ignore_index=True)
checks.banner(allq)
print(len(allq), "answers in", len(SETTINGS), "settings")

## Step 1 · How often is *unanimous* still wrong?
Absolute definition: all 10 samples identical. If agreement guaranteed correctness, the last column would be 0%.

In [ ]:
g = allq.groupby("setting")
tab = pd.DataFrame({
    "unanimous": g.apply(lambda d: int((d.U_sc == 0).sum())),
    "unanimous_wrong": g.apply(lambda d: int(((d.U_sc == 0) & (d.label_containment == 0)).sum())),
})
tab["share_wrong"] = (tab.unanimous_wrong / tab.unanimous.replace(0, np.nan)).round(3)
tab

## Step 2 · Catalogue the failures
Pick **up to 10** confident-but-wrong cases. For each, decide the cause:

* `fabrication`: the model really is wrong
* `strict_gold`: the answer is right but the gold string is too strict (e.g. "doctor" vs "physician")
* `label_noise`: the label is wrong (e.g. wrong answer credited because it mentions the gold string)
* `ambiguous`: reasonable people disagree

In [ ]:
cases = allq[(allq.U_sc == 0) & (allq.label_containment == 0)].head(10)
cases[["setting", "qid", "question", "greedy", "gold", "label_judge"]]

In [ ]:
# Edit this dictionary with your group's verdicts, then run the next cell
my_verdicts = {
    # "qid": "fabrication",
}
print(pd.Series(my_verdicts).value_counts() if my_verdicts else "Add some verdicts above.")

## Step 3 · Where do the two labelings disagree?
`label_containment` is a string rule; `label_judge` comes from an LLM judge. Disagreements show how label noise moves results.

In [ ]:
dis = allq[allq.label_containment != allq.label_judge]
print(f"{len(dis)} of {len(allq)} answers ({len(dis)/len(allq):.1%}) get different labels")
dis[["setting", "question", "greedy", "gold", "label_containment", "label_judge"]].head(8)

## Step 4 · The detector's most confident mistakes
Wrong answers with the **lowest** Comb4 uncertainty (the system would answer these without hesitation).

In [ ]:
worst = allq[allq.label_containment == 0].nsmallest(10, "Comb4")
worst[["setting", "question", "greedy", "gold", "U_sc", "U_se", "U_lp", "U_nli", "Comb4"]].round(2)

## Discussion (for the last 8 minutes)
1. Which failures could a **better detector** fix, and which need **outside evidence** (retrieval, a database, a human)?
2. How much of what looks like "model error" is actually **label error**?
3. Your product answers medical or legal questions. Which **coverage** would you accept, and who handles the abstentions?
4. Would you add a fifth sampling signal, or something of a different kind?